In [45]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import MinMaxScaler

import torch
from transformers import pipeline, CLIPModel, CLIPProcessor
from PIL import Image
import matplotlib.pyplot as plt

import pandas as pd

### Dataset generation

In [2]:
n_vectors = 1000
vector_len = 8
n_blocks = 4
block_len = vector_len // n_blocks
noise_level = 0.3

In [3]:
seed = 42
np.random.seed(seed)

cluster_centers = np.empty((n_blocks, block_len))
for cluster in range(n_blocks):
    cluster_centers[cluster] = np.random.uniform(-1, 1, block_len) + cluster*2

vectors_dataset = np.zeros((n_vectors, vector_len))

for i in range(n_vectors):
    for block in range(n_blocks):
        vectors_dataset[i][(block_len * block) : (block_len * (block+1))] = cluster_centers[block] + np.random.normal(0, noise_level, block_len)

vectors_dataset[0:3]

array([[ 0.22284408,  1.13165903,  2.32314557,  2.36008498,  3.17301197,
         3.17227011,  5.18875591,  6.15836822],
       [-0.76839511,  0.73274235,  2.16013855,  2.29159117,  3.03963006,
         2.88829793,  5.55586186,  6.6646194 ],
       [-0.2306613 ,  0.47400416,  2.30067307,  2.23059375,  2.96673921,
         3.42469845,  4.93597562,  6.64484417]])

### Vanilla product quantization (with K-Means clustering)

In [4]:
def product_quantize(vectors, 
                     n_blocks: int = 4, 
                     n_clusters: int = 1,
                     seed: int = 42, 
                     cluster_init: int = 3):
    block_len = vectors.shape[1] // n_blocks
    subspaces = []
    cluster_labels = []
    cluster_centroids = []
    subspaces_q = []

    for i in range(0, vectors.shape[1] - block_len + 1, block_len):
        subspaces.append(vectors[:, i:(i + block_len)])

    for block in range(n_blocks):
        kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=cluster_init)
        cluster_labels.append(kmeans.fit_predict(subspaces[block]))
        cluster_centroids.append(kmeans.cluster_centers_)

    for i in range(n_blocks):
        subspaces_q.append( cluster_centroids[i][cluster_labels[i]] )
    
    return np.concat(subspaces_q, axis = 1)   


In [5]:
vectors_quantized = product_quantize(vectors_dataset, n_blocks, 2, seed, cluster_init=3)
print("Product quantization RMSE for randomly generated vectors: ", root_mean_squared_error(vectors_dataset, vectors_quantized))

Product quantization RMSE for randomly generated vectors:  0.24553920033432028


### Embedding extraction

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

clip = clip.eval()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [7]:
def get_image_embeddings(model, images_paths: list, device: torch.device):
    images = [Image.open(image_path) for image_path in images_paths]
    inputs = processor(images=images, return_tensors="pt").to(device)
    
    with torch.no_grad():
        image_features = model.get_image_features(**inputs)
        image_embeddings = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
    
    return image_embeddings.cpu().numpy().astype(np.float32)

In [8]:
images_folder = "./images/"
images_names = ["cat.jpg", "dog.jpg", "wolf.png", "gosling.png"]

In [9]:
image_embeddings = get_image_embeddings(
    clip, 
    [images_folder + image_name for image_name in images_names], 
    device
) 
print(f"Embeddings of images {images_names} have shape:  {image_embeddings.shape}")

Embeddings of images ['cat.jpg', 'dog.jpg', 'wolf.png', 'gosling.png'] have shape:  (4, 512)


In [10]:
image_embeddings[0][:10]

array([-0.00465205, -0.01560188,  0.01688051,  0.02128985,  0.01414021,
       -0.02529875, -0.01026839,  0.04886029,  0.00564997,  0.00126772],
      dtype=float32)

### Vanilla product quantization application to embeddings

In [11]:
embeddings_quantized = product_quantize(image_embeddings, n_blocks, 1, seed, cluster_init=3)
print("Product quantization RMSE for CLIP embeddings: ", root_mean_squared_error(image_embeddings, embeddings_quantized))

Product quantization RMSE for CLIP embeddings:  0.023134570568799973


### Comparison of data being quantized

In [24]:
vectors_df = pd.DataFrame(vectors_dataset)
embedds_df = pd.DataFrame(image_embeddings)

In [ ]:
vectors_df.describe()

,0,1,2,3,4,5,6,7
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.255657,0.910206,2.462761,2.188165,3.300917,3.307466,5.133625,6.728513
std,0.310334,0.307195,0.300245,0.300534,0.296850,0.302795,0.294550,0.304438
min,-1.223300,0.025023,1.558134,1.283920,2.425763,2.205479,4.149862,5.652027
25%,-0.456807,0.706571,2.262167,1.983706,3.100352,3.101743,4.934108,6.516315
50%,-0.255817,0.911341,2.463527,2.185711,3.294620,3.310724,5.134828,6.729997
75%,-0.054177,1.107393,2.662008,2.392507,3.502901,3.517166,5.332964,6.925029
max,0.694697,2.079300,3.344685,3.353136,4.196766,4.370706,6.089095,7.673677


In [25]:
embedds_df.describe()

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
count,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000
mean,-0.017586,0.006892,0.004470,0.023240,0.003722,-0.055876,0.027875,0.005301,-0.022363,0.007100,...,-0.007254,0.023876,0.006119,0.026009,-0.015105,-0.018923,0.009336,0.023863,-0.010809,0.011633
std,0.034372,0.015017,0.023116,0.014786,0.013657,0.039715,0.032737,0.036716,0.019312,0.025122,...,0.032741,0.021806,0.007690,0.029687,0.019524,0.036225,0.016783,0.014190,0.024458,0.016080
min,-0.049503,-0.015602,-0.027159,0.007923,-0.010150,-0.113338,-0.010268,-0.028546,-0.038071,-0.020241,...,-0.053525,0.000676,-0.003733,-0.004999,-0.031701,-0.055657,-0.010440,0.002599,-0.044363,-0.008765
25%,-0.043197,0.006411,-0.004847,0.017199,-0.006879,-0.066291,0.007264,-0.023183,-0.032716,-0.004110,...,-0.018029,0.009369,0.001969,0.014332,-0.029798,-0.047234,-0.000743,0.023208,-0.018463,0.003626
50%,-0.022874,0.013827,0.009736,0.020790,0.004176,-0.042435,0.029500,0.000445,-0.028516,0.004099,...,0.002040,0.022478,0.007510,0.021281,-0.019477,-0.016209,0.010002,0.030658,-0.006317,0.013263
75%,0.002737,0.014308,0.019052,0.026832,0.014777,-0.032020,0.050112,0.028928,-0.018162,0.015309,...,0.012815,0.036986,0.011660,0.032958,-0.004783,0.012103,0.020080,0.031313,0.001336,0.021270
max,0.024905,0.015517,0.025568,0.043457,0.016687,-0.025299,0.062766,0.048860,0.005650,0.040443,...,0.020432,0.049871,0.013189,0.066471,0.010237,0.012383,0.027779,0.031537,0.013761,0.028769


In [42]:
def analyze_magnitude(num_df : pd.DataFrame,
                      num_df_name : str = "dataframe"):
    num_df_min = num_df.min().min()
    num_df_max = num_df.max().max()
    print(f"Magnitude of {num_df_name}: [ {num_df_min:.4f} ; {num_df_max:.4f} ] => \
    the interval length is  {num_df_max - num_df_min:.4f}")

In [51]:
analyze_magnitude(vectors_df, "generated vector dataset")
analyze_magnitude(embedds_df, "         CLIP embeddings")

Magnitude of generated vector dataset: [ -1.2233 ; 7.6737 ] =>     the interval length is  8.8970
Magnitude of          CLIP embeddings: [ -0.6616 ; 0.2335 ] =>     the interval length is  0.8951


In [50]:
vectors_sc = MinMaxScaler().fit_transform(vectors_df)
embedds_sc = MinMaxScaler().fit_transform(embedds_df)
analyze_magnitude(vectors_sc, " scaled vector dataset")
analyze_magnitude(embedds_sc, "scaled CLIP embeddings")

Magnitude of  scaled vector dataset: [ 0.0000 ; 1.0000 ] =>     the interval length is  1.0000
Magnitude of scaled CLIP embeddings: [ 0.0000 ; 1.0000 ] =>     the interval length is  1.0000


In [52]:
seed = 42

vectors_sc_quantized = product_quantize(vectors_sc, n_blocks, 2, seed, cluster_init=3)
print("Product quantization RMSE for randomly generated vectors (scaled): ", root_mean_squared_error(vectors_sc, vectors_sc_quantized))
embedds_sc_quantized = product_quantize(embedds_sc, n_blocks, 2, seed, cluster_init=3)
print("Product quantization RMSE for CLIP embeddings (scaled):            ", root_mean_squared_error(embedds_sc, embedds_sc_quantized))

Product quantization RMSE for randomly generated vectors (scaled):  0.12285585663244178
Product quantization RMSE for CLIP embeddings (scaled):             0.27493107318878174


**Вывод о большей RMSE у хорошо кластеризуемых данных:** 

Длина интервала значений сгенерированного датасета примерно в 10 раз меньше таковой эмбеддингов, что вызывает разницу сопоставимого масштаба в соответствующих RMSE. Объяснение выдерживает проверку шкалированием обоих интервалов: при совпадающей длине синтетический хорошо кластеризуемый датасет восстанавливается из квантованного с меньшей ошибкой реконструкции.